In [1]:
%pip install langchain>=0.3 langchain-community>=0.3 langchain-openai>=0.2 langchain-text-splitters>=0.3 faiss-cpu>=1.8 openai>=1.40 python-dotenv>=1.0 scikit-learn>=1.4 langsmith

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import uuid
from langsmith import Client
from langsmith.evaluation import evaluate
from dotenv import load_dotenv

# Cargar las variables desde el archivo .env
load_dotenv()

# Verificamos que se haya cargado correctamente la clave principal
if not os.getenv("LLM_API_KEY"):
    print("⚠️ ADVERTENCIA: No se encontró LLM_API_KEY. Revisa tu archivo .env")
else:
    print("✅ Variables de entorno cargadas correctamente.")

✅ Variables de entorno cargadas correctamente.


In [3]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
import os

# 1. Ajuste clave: Leemos específicamente la variable MISTRAL_EMBED_KEY
MISTRAL_API_KEY = os.getenv("MISTRAL_EMBED_KEY")
MISTRAL_BASE_URL = os.getenv("MISTRAL_EMBED_BASE_URL", "https://api.mistral.ai/v1")

embeddings_mistral = OpenAIEmbeddings(
    api_key=MISTRAL_API_KEY,
    base_url=MISTRAL_BASE_URL,
    model="mistral-embed",
    check_embedding_ctx_length=False
)

docs_simulados = [
    Document(page_content="Reglamento Art 12: Las convalidaciones se piden los primeros 10 días hábiles.", metadata={"source": "reglamento.md"}),
    Document(page_content="Asignatura Ingeniería de Software (código INF301), semestre 3, 10 créditos. Prerrequisito: Programación Avanzada.", metadata={"source": "malla.csv"}),
    Document(page_content="Las becas de excelencia académica se postulan hasta el 30 de marzo.", metadata={"source": "becas.md"})
]

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = splitter.split_documents(docs_simulados)

vector_db = FAISS.from_documents(chunks, embeddings_mistral)

class InternalRetrievalTool:
    def build_context(self, query: str) -> str:
        results = vector_db.similarity_search(query, k=2)
        return "\n\n".join(f"[{doc.metadata['source']}] {doc.page_content}" for doc in results)

print("¡Base de datos FAISS creada exitosamente leyendo la clave de Mistral del .env!")

C:\Users\Yoyi\AppData\Local\Temp\ipykernel_21712\3033957697.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


¡Base de datos FAISS creada exitosamente leyendo la clave de Mistral del .env!


In [4]:
INTENT_CLASSIFIER_PROMPT = """Eres el clasificador de la Secretaría Académica.
Categorías: malla, reglamento, becas_calendario, normativa_externa, fuera_de_alcance.
Ejemplo: "¿Hasta cuándo postulo a la beca?" -> becas_calendario
Ejemplo: "Quiero apelar mi nota" -> fuera_de_alcance
Consulta: "{query}"
Categoría:"""

RAG_ANSWER_PROMPT = """Responde usando EXCLUSIVAMENTE el contexto. 
Si no hay información, dilo. Cita la fuente al final de la afirmación entre corchetes.
CONTEXTO: {context}
CONSULTA: {query}
RESPUESTA:"""

ESCALATION_CHECK_PROMPT = """Responde 'ESCALAR' si la consulta involucra apelaciones, salud, temas legales o socioeconómicos. Si no, responde 'NO_ESCALAR'.
Consulta: "{query}"
Respuesta:"""

In [5]:
# ==============================================================================
# PROMPTS Y AGENTE ORQUESTADOR CON FALLBACK DE ALTA DISPONIBILIDAD
# ==============================================================================
from langchain_openai import ChatOpenAI
import os

RAG_ANSWER_PROMPT = """Eres el asistente de la Secretaría Académica.
Responde a la CONSULTA del estudiante basándote EXCLUSIVAMENTE en el CONTEXTO proporcionado.
Cita la fuente al final entre corchetes.

CONTEXTO:
{context}

CONSULTA: {query}
RESPUESTA:"""

class Orchestrator:
    def __init__(self):
        self.llm = ChatOpenAI(
            api_key=os.getenv("LLM_API_KEY"),
            base_url=os.getenv("LLM_BASE_URL", "https://api.mistral.ai/v1"),
            model=os.getenv("LLM_MODEL", "mistral-small-latest"),
            temperature=0.1,
            max_retries=1
        )
        self.retrieval = InternalRetrievalTool()
        self.high_risk_keywords = ["apelar", "abogado", "demanda", "salud mental", "depresión", "datos personales"]

    def _safe_invoke(self, prompt: str):
        """Invoca al LLM; si la API está bloqueada por Rate Limit (429), retorna None para activar el fallback."""
        try:
            return self.llm.invoke(prompt).content
        except Exception as e:
            print(f"⚠️ API saturada ({e}). Activando fallback local de resiliencia...")
            return None

    def handle(self, query: str) -> dict:
        # 1. Filtro local de seguridad de alto riesgo (0 llamadas a la API)
        if any(kw in query.lower() for kw in self.high_risk_keywords):
            return {
                "respuesta": "Tu consulta requiere revisión de un funcionario humano. Hemos generado un ticket.",
                "escalado": True
            }

        # 2. Recuperar contexto desde FAISS
        context = self.retrieval.build_context(query)

        # 3. Intentar respuesta con el LLM
        rag_prompt = RAG_ANSWER_PROMPT.format(context=context, query=query)
        respuesta_llm = self._safe_invoke(rag_prompt)

        # 4. Fallback automático si la API responde 429
        if respuesta_llm is None:
            respuesta_final = f"Información encontrada en sistema: {context}"
        else:
            respuesta_final = respuesta_llm

        return {
            "respuesta": respuesta_final,
            "escalado": False
        }

print("¡Agente Orquestador con Fallback de Alta Disponibilidad cargado!")

¡Agente Orquestador con Fallback de Alta Disponibilidad cargado!


In [6]:
# ==============================================================================
# EVALUACIÓN LANGSMITH (Rápida y Resiliente)
# ==============================================================================
import uuid
from langsmith import Client
from langsmith.evaluation import evaluate

agente = Orchestrator()
client = Client()

dataset_name = f"Evaluacion_Secretaria_Academica_{uuid.uuid4().hex[:4]}"
dataset = client.create_dataset(dataset_name, description="Set de evaluación de escalamiento y RAG")

ejemplos = [
    {"inputs": {"query": "Quiero apelar mi nota de grado"}, "outputs": {"escalado": True}},
    {"inputs": {"query": "Tengo depresión y no puedo rendir"}, "outputs": {"escalado": True}},
    {"inputs": {"query": "¿Cuándo postulo a las becas?"}, "outputs": {"escalado": False}}
]

for ex in ejemplos:
    client.create_example(inputs=ex["inputs"], outputs=ex["outputs"], dataset_id=dataset.id)

def agente_wrapper(inputs: dict) -> dict:
    print(f"Procesando: '{inputs['query']}'")
    resultado = agente.handle(inputs["query"])
    return {"escalado": resultado["escalado"], "respuesta": resultado["respuesta"]}

def exact_escalation_match(run, example) -> dict:
    esperado = example.outputs["escalado"]
    obtenido = run.outputs.get("escalado") if run.outputs else None 
    return {"key": "Escalamiento Correcto", "score": int(esperado == obtenido) if obtenido is not None else 0}

print(f"Iniciando evaluación en el dataset: {dataset_name}...")
results = evaluate(
    agente_wrapper,
    data=dataset_name,
    evaluators=[exact_escalation_match],
    client=client,
    max_concurrency=1
)
print("¡Evaluación completada con éxito!")

Iniciando evaluación en el dataset: Evaluacion_Secretaria_Academica_7250...
View the evaluation results for experiment: 'healthy-wrist-26' at:
https://smith.langchain.com/o/e38da54f-ddc6-4531-8d09-d3394427f9c1/datasets/4ede29ae-0319-4056-9e0e-58364c48891c/compare?selectedSessions=9082c078-4206-4858-ab6b-126633fc94d8


Procesando: '¿Cuándo postulo a las becas?'
⚠️ API saturada (Error code: 429 - {'object': 'error', 'message': 'Rate limit exceeded', 'type': 'rate_limited', 'param': None, 'code': '1300', 'raw_status_code': 429}). Activando fallback local de resiliencia...
Procesando: 'Tengo depresión y no puedo rendir'
Procesando: 'Quiero apelar mi nota de grado'
¡Evaluación completada con éxito!
